# Лабораторна робота №2 - Частина 2
## Дослідження Individual Household Electric Power Consumption Dataset

**Мета:** Використання `pandas`, `timeit` для обробки великих наборів даних, обчислення кореляцій та One Hot Encoding.

In [1]:
import pandas as pd
import numpy as np
import timeit
import warnings

warnings.filterwarnings('ignore')

# Допоміжна функція для профілювання часу (вимоги до timeit)
def profile_execution(func, *args, **kwargs):
    start = timeit.default_timer()
    result = func(*args, **kwargs)
    end = timeit.default_timer()
    print(f"[{func.__name__}] Час виконання: {end - start:.5f} сек.")
    return result

### 1. Зчитування та Data Cleaning
Завантажуємо датасет, перетворюємо '?' на NaN, видаляємо пропуски, зводимо дату та час у зручний формат.

In [2]:
def load_and_clean_data(filepath='household_power_consumption.txt'):
    # Зчитування з обробкою '?' як NaN
    df = pd.read_csv(filepath, sep=';', na_values=['?'], 
                     dtype={'Global_active_power': float, 'Global_reactive_power': float, 
                            'Voltage': float, 'Global_intensity': float, 
                            'Sub_metering_1': float, 'Sub_metering_2': float, 'Sub_metering_3': float})
    
    # Видалення рядків з NaN
    df.dropna(inplace=True)
    
    # Створення зручної колонки Datetime
    df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
    
    return df

print("Завантаження даних...")
df_power = profile_execution(load_and_clean_data)
print("Дані завантажено успішно. Розмір:", df_power.shape)

Завантаження даних...
[load_and_clean_data] Час виконання: 7.08112 сек.
Дані завантажено успішно. Розмір: (2049280, 10)


### 2. Формування вибірок
Реалізація чотирьох специфічних запитів. Кожна функція обгорнута у профілювальник часу.

In [3]:
def query_1_high_power(df):
    """1. Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт."""
    return df[df['Global_active_power'] > 5.0]

def query_2_intensity_and_metering(df):
    """2. Сила струму в межах 19-20 А, пральна машина та холодильник (Sub_metering_2) 
    споживають більше, ніж бойлер та кондиціонер (Sub_metering_3)."""
    mask = (df['Global_intensity'] >= 19.0) & (df['Global_intensity'] <= 20.0) & \
           (df['Sub_metering_2'] > df['Sub_metering_3'])
    return df[mask]

def query_3_random_sample_means(df):
    """3. Випадково 500000 записів (без повторів), обчислити середні величини 3-х груп споживання."""
    sample = df.sample(n=500000, replace=False, random_state=42)
    means = sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means

def query_4_complex_evening_filter(df):
    """4. Після 18-00 споживають понад 6 кВт за хв. Sub_metering_2 є найбільшим.
    Потім обрати кожен третій результат із першої половини та кожен четвертий із другої."""
    # Відбираємо після 18:00 і > 6 кВт
    filtered = df[(df['Datetime'].dt.hour >= 18) & (df['Global_active_power'] > 6.0)]
    
    # Відбираємо ті, де 2 група найбільша
    mask_group2_biggest = (filtered['Sub_metering_2'] > filtered['Sub_metering_1']) & \
                          (filtered['Sub_metering_2'] > filtered['Sub_metering_3'])
    filtered = filtered[mask_group2_biggest]
    
    # Ділимо навпіл
    half_idx = len(filtered) // 2
    first_half = filtered.iloc[:half_idx]
    second_half = filtered.iloc[half_idx:]
    
    # Кожен 3-й з першої, кожен 4-й з другої
    res1 = first_half.iloc[2::3]
    res2 = second_half.iloc[3::4]
    
    return pd.concat([res1, res2])

# --- Виклики та профілювання ---
print("\n--- Результати вибірок ---")
res1 = profile_execution(query_1_high_power, df_power)
print(f"Знайдено записів (Потужність > 5): {len(res1)}")

res2 = profile_execution(query_2_intensity_and_metering, df_power)
print(f"Знайдено записів (Струм 19-20А, Sub2 > Sub3): {len(res2)}")

res3 = profile_execution(query_3_random_sample_means, df_power)
print(f"Середні значення (Випадкові 500k):\n{res3.to_string()}")

res4 = profile_execution(query_4_complex_evening_filter, df_power)
print(f"Знайдено записів (Вечірні складні фільтри): {len(res4)}")


--- Результати вибірок ---
[query_1_high_power] Час виконання: 0.00573 сек.
Знайдено записів (Потужність > 5): 17547
[query_2_intensity_and_metering] Час виконання: 0.01047 сек.
Знайдено записів (Струм 19-20А, Sub2 > Sub3): 2509
[query_3_random_sample_means] Час виконання: 0.20855 сек.
Середні значення (Випадкові 500k):
Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
[query_4_complex_evening_filter] Час виконання: 0.05443 сек.
Знайдено записів (Вечірні складні фільтри): 308


### 3. Нормування, Стандартизація та Кореляція

In [4]:
def normalize_and_standardize(df, column):
    """Пронормувати та стандартизувати датасет (на прикладі Global_active_power)"""
    # Нормалізація (Min-Max)
    normalized = (df[column] - df[column].min()) / (df[column].max() - df[column].min())
    # Стандартизація (Z-score)
    standardized = (df[column] - df[column].mean()) / df[column].std()
    return normalized, standardized

norm, stand = normalize_and_standardize(df_power, 'Global_active_power')
print("Приклад нормалізації (перші 3):\n", norm.head(3).values)
print("Приклад стандартизації (перші 3):\n", stand.head(3).values)

# Підрахунок коефіцієнтів кореляції
col1, col2 = 'Global_active_power', 'Global_intensity'

pearson_corr = df_power[col1].corr(df_power[col2], method='pearson')
spearman_corr = df_power[col1].corr(df_power[col2], method='spearman')

print(f"\nКоефіцієнт Пірсона між {col1} та {col2}: {pearson_corr:.4f}")
print(f"Коефіцієнт Спірмена між {col1} та {col2}: {spearman_corr:.4f}")

Приклад нормалізації (перші 3):
 [0.37479631 0.47836321 0.47963064]
Приклад стандартизації (перші 3):
 [2.95507634 4.03708364 4.05032499]

Коефіцієнт Пірсона між Global_active_power та Global_intensity: 0.9989
Коефіцієнт Спірмена між Global_active_power та Global_intensity: 0.9954


### 4. One Hot Encoding категоріального атрибута
Оскільки явно категоріальних текстових атрибутів немає, створимо колонку "День тижня" з Datetime та застосуємо One Hot Encoding до неї.

In [5]:
# Створюємо категоріальний атрибут (Назва дня тижня)
df_power['DayOfWeek'] = df_power['Datetime'].dt.day_name()

# Застосовуємо One Hot Encoding (створюємо dummy змінні)
df_encoded = pd.get_dummies(df_power, columns=['DayOfWeek'], dtype=int)

print("Колонки після One Hot Encoding:")
# Виводимо останні 8 колонок для наочності (саме там з'являться нові)
display(df_encoded.iloc[:, -8:].sample(5))

Колонки після One Hot Encoding:


,Datetime,DayOfWeek_Friday,DayOfWeek_Monday,DayOfWeek_Saturday,DayOfWeek_Sunday,DayOfWeek_Thursday,DayOfWeek_Tuesday,DayOfWeek_Wednesday
1063591,2008-12-24 07:55:00,0,0,0,0,0,0,1
453528,2007-10-27 16:12:00,0,0,1,0,0,0,0
1326191,2009-06-24 16:35:00,0,0,0,0,0,0,1
1627369,2010-01-19 20:13:00,0,0,0,0,0,1,0
1508511,2009-10-29 07:15:00,0,0,0,0,1,0,0
